# Chess Lesson — configure cell 1 then run all
Analysis and PDF are generated by `lesson_runner.py`. This notebook displays inline SVG boards for reference.

In [ ]:
from lesson_runner import run_lesson, most_common_sequence
import chess, chess.svg
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── CONFIGURE FOR EACH LESSON ─────────────────────────────────────────────────
config = {
    'username':      'sfink37',
    'games_dir':     'games_sf',
    'time_classes':  {'bullet', 'blitz'},
    'eco_keywords':  ['Caro', 'Advance'],
    'lesson_title':  'Caro-Kann Defense Advance (as White)',
    'lesson_number': 1,
}
# ─────────────────────────────────────────────────────────────────────────────

BOARD_SIZE = 380

def show_board(board, arrows=None, caption=''):
    svg  = chess.svg.board(board, arrows=arrows or [], size=BOARD_SIZE)
    html = svg
    if caption:
        html = (f'<p style="font-family:monospace;font-size:13px;'
                f'margin:4px 0 8px 0;color:#555">{caption}</p>') + html
    display(HTML(html))

In [ ]:
# Run full analysis + save PDF
results = run_lesson(config)

games      = results['games']
consistent = results['consistent']
sf_evals   = results['sf_evals']
print(f"\nPDF: {results['pdf_path']}")

## Inline reference — junction visualizations
SVG boards below are for interactive reference. The PDF uses the matplotlib renderer.

In [ ]:
for idx, (jt, jfen, jstats) in enumerate(consistent):
    jstats_sorted = sorted(jstats, key=lambda x: -x[2])
    jcp     = sf_evals.get(jfen, {})
    best_cp = max(jcp.values()) if jcp else None

    # Bar chart
    labels  = [s[0] for s in jstats_sorted]
    wrs     = [s[2] for s in jstats_sorted]
    ns      = [s[1] for s in jstats_sorted]
    colors  = ['#27ae60' if w >= 45 else '#c0392b' if w < 35 else '#e67e22' for w in wrs]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars    = ax.bar(labels, wrs, color=colors, width=0.5, edgecolor='white')
    ax.axhline(50, color='gray', linestyle='--', linewidth=1)
    ax.set_ylabel('Win %')
    ax.set_title(f'Junction {idx+1} of {len(consistent)} — {jt} games')
    ax.set_ylim(0, 90)
    for bar, wv, n in zip(bars, wrs, ns):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{wv:.0f}%\n({n}g)', ha='center', va='bottom', fontsize=10)
    ax.legend(handles=[
        mpatches.Patch(color='#27ae60', label='>= 45%  good'),
        mpatches.Patch(color='#e67e22', label='35-44%  ok'),
        mpatches.Patch(color='#c0392b', label='< 35%  avoid'),
    ], loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()

    # SVG board with colored arrows
    jboard = chess.Board(jfen + ' w KQkq - 0 1')
    arrows = []
    for san, t, wr, c in jstats_sorted:
        col = '#27ae60' if wr >= 45 else '#c0392b' if wr < 35 else '#e67e22'
        try:
            move_obj = jboard.parse_san(san)
            arrows.append(chess.svg.Arrow(move_obj.from_square,
                                          move_obj.to_square, color=col))
        except Exception:
            pass
    show_board(jboard, arrows=arrows,
               caption=f'Junction {idx+1} ({jt} games) — green good | orange ok | red avoid')

    # Move sequence
    seq = most_common_sequence(games, jfen)
    if seq:
        print(f'\n  Route to junction — most common line:')
        print(f'  {"#":<5} {"White":<12} Black')
        print('  ' + '-' * 30)
        for i in range(0, len(seq), 2):
            mn = i // 2 + 1
            wm = seq[i]
            bm = seq[i+1] if i+1 < len(seq) else ''
            print(f'  {mn:<5} {wm:<12} {bm}')
        print('  --> white to move')

    # Stats table
    print(f'\n  {"Move":<8} {"Win%":>6}  {"SF delta":>9}  Record')
    print('  ' + '-' * 46)
    for san, t, wr, c in jstats_sorted:
        cp    = jcp.get(san)
        delta = (cp - best_cp) if (cp is not None and best_cp is not None) else None
        ds    = f'{delta:+d}cp' if delta is not None else '---'
        tag   = '  [GOOD]' if wr >= 45 else '  [AVOID]' if wr < 35 else '  [OK]'
        print(f'  {san:<8} {wr:>5.0f}%  {ds:>9}  {t}g W:{c["win"]} L:{c["loss"]} D:{c.get("draw",0)}{tag}')
    print()